![image](https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use watsonx, and `ibm/granite-4-h-small` to support multiple languages translation

#### Disclaimers

- Use only Projects and Spaces that are available in watsonx context.


## Notebook content

This notebook contains the steps and code to demonstrate support for language translation in watsonx. It introduces commands for defining prompt and model testing.

Some familiarity with Python is helpful. This notebook uses Python 3.11.

## Learning goal

The goal of this notebook is to demonstrate how to translate multiple languages using `ibm/granite-4-h-small` watsonx model based on query provided by the user.


## Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [Foundation Models on watsonx.ai](#Foundation-Models-on-watsonx.ai)
3. [Translate the text based on the query](#Translate-the-text-based-on-the-query)
4. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).


### Install dependencies

In [1]:
%pip install -U ibm-watsonx-ai | tail -n 1

Note: you may need to restart the kernel to use updated packages.


### Defining the watsonx.ai credentials
This cell defines the watsonx.ai credentials required to work with watsonx Foundation Model inferencing.

**Action:** Provide the IBM Cloud user API key. For details, see <a href="https://cloud.ibm.com/docs/account?topic=account-userapikey&interface=ui" target="_blank" rel="noopener no referrer">documentation</a>.

In [2]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Please enter your watsonx.ai api key (hit enter): "),
)

### Defining the project ID
The Foundation Model requires project ID that provides the context for the call. We will obtain the ID from the project in which this notebook runs. Otherwise, please provide the project ID.

In [3]:
import os

try:
    project_id = os.environ["PROJECT_ID"]
except KeyError:
    project_id = input("Please enter your project_id (hit enter): ")

### API Client initialization

In [4]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials, project_id=project_id)

<a id="Foundation-Models-on-watsonx.ai"></a>
## Foundation Models on watsonx.ai

#### List available models

In [5]:
client.foundation_models.TextModels.show()

{'GRANITE_3_2_8B_INSTRUCT': 'ibm/granite-3-2-8b-instruct', 'GRANITE_3_2B_INSTRUCT': 'ibm/granite-3-2b-instruct', 'GRANITE_3_3_8B_INSTRUCT': 'ibm/granite-3-3-8b-instruct', 'GRANITE_3_8B_INSTRUCT': 'ibm/granite-3-8b-instruct', 'GRANITE_4_H_SMALL': 'ibm/granite-4-h-small', 'GRANITE_8B_CODE_INSTRUCT': 'ibm/granite-8b-code-instruct', 'GRANITE_GUARDIAN_3_8B': 'ibm/granite-guardian-3-8b', 'GRANITE_VISION_3_2_2B': 'ibm/granite-vision-3-2-2b', 'LLAMA_3_2_11B_VISION_INSTRUCT': 'meta-llama/llama-3-2-11b-vision-instruct', 'LLAMA_3_2_90B_VISION_INSTRUCT': 'meta-llama/llama-3-2-90b-vision-instruct', 'LLAMA_3_3_70B_INSTRUCT': 'meta-llama/llama-3-3-70b-instruct', 'LLAMA_3_405B_INSTRUCT': 'meta-llama/llama-3-405b-instruct', 'LLAMA_4_MAVERICK_17B_128E_INSTRUCT_FP8': 'meta-llama/llama-4-maverick-17b-128e-instruct-fp8', 'LLAMA_GUARD_3_11B_VISION': 'meta-llama/llama-guard-3-11b-vision', 'MISTRAL_MEDIUM_2505': 'mistralai/mistral-medium-2505', 'MISTRAL_SMALL_3_1_24B_INSTRUCT_2503': 'mistralai/mistral-small-3

You need to specify `model_id` that will be used for inferencing:

In [6]:
model_id = client.foundation_models.TextModels.GRANITE_4_H_SMALL

### Defining the model parameters

You might need to adjust model `parameters` for different models or tasks, to do so please refer to <a href="https://ibm.github.io/watsonx-ai-python-sdk/fm_model.html#metanames.GenTextParamsMetaNames" target="_blank" rel="noopener no referrer">documentation</a>.

In [7]:
from ibm_watsonx_ai.foundation_models.utils.enums import DecodingMethods
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams

parameters = {
    GenParams.DECODING_METHOD: DecodingMethods.SAMPLE,
    GenParams.MAX_NEW_TOKENS: 100,
    GenParams.MIN_NEW_TOKENS: 1,
    GenParams.TEMPERATURE: 0.5,
    GenParams.TOP_K: 50,
    GenParams.TOP_P: 1,
    GenParams.STOP_SEQUENCES: ["\n"],
}

**Warning:** Delete `GenParams.STOP_SEQUENCES: ["\n"]` parameter if you intend to utilize the model for multiple translations in one prompt.

### Initialize the model
Initialize the `Model` class with previous set params.

In [8]:
from ibm_watsonx_ai.foundation_models import ModelInference

model = ModelInference(
    model_id=model_id, params=parameters, credentials=credentials, project_id=project_id
)

### Model's details

In [9]:
import json

print(json.dumps(model.get_details(), indent=2))

{
  "model_id": "ibm/granite-4-h-small",
  "label": "granite-4-h-small",
  "provider": "IBM",
  "source": "IBM",
  "indemnity": "IBM_COVERED",
  "functions": [
    {
      "id": "autoai_rag"
    },
    {
      "id": "text_chat"
    },
    {
      "id": "text_generation"
    }
  ],
  "short_description": "Granite-4.0-H-Small is a 30B parameter long-context instruct model finetuned from Granite-4.0-H-Small-Base using a combination of open source instruction datasets with permissive license and internally collected synthetic datasets.",
  "long_description": "Granite-4.0-H-Small is a 30B parameter long-context instruct model finetuned from Granite-4.0-H-Small-Base using a combination of open source instruction datasets with permissive license and internally collected synthetic datasets. This model is developed using a diverse set of techniques with a structured chat format, including supervised finetuning, model alignment using reinforcement learning, and model merging. Granite 4.0 instru

<a id="Translate-the-text-based-on-the-query"></a>
## Translate the text based on the query

### English to Spanish translation:

Define query for the model with at least one example, specifically for English to Spanish translation.

**Note:** Model works the best with at least one translation example.

In [10]:
english_to_spanish_query = """Translate the following text from English to Spanish:

Input: So far, I have not been terribly encouraged by the stance adopted by the Commission.
Output: Hasta ahora no me ha animado mucho la postura adoptada por la Comisión.

Input: I am very pleased to see that the joint resolution adopts the suggestion we made.
"""

**Warning:** ensure that there is a line break (newline) at the conclusion of the prompt.

### Generate the English to Spanish translation using `ibm/granite-4-h-small` model.

In [11]:
translation_result = model.generate_text(english_to_spanish_query)

Print the translation result

In [12]:
print(translation_result)

Output: Me alegra mucho ver que la resolución conjunta adopta la sugerencia que hicimos.



### English to French translation:

Define query for the model with at least one example, specifically for English to French translation.

**Note:** Model works the best with at least one translation example.

In [13]:
english_to_french_query = """Translate the following text from English to French:

Input: Finally, I welcome paragraph 16 which calls for a review of the way we deal with human rights issues in Parliament.
Output: Enfin, je me réjouis du paragraphe 16 qui appelle à une révision de la manière dont nous abordons les questions relatives aux droits de l'homme au sein du Parlement.

Input: I remember very well that we discussed it in a session in Luxembourg.
Output: Je me souviens très bien que nous en avions parlé lors d'une séance à Luxembourg.

Input: If we do not greatly increase the use of intelligent technology, we will not achieve our targets.
"""

**Warning:** ensure that there is a line break (newline) at the conclusion of the prompt.

### Generate the English to French translation using `ibm/granite-4-h-small` model.

In [14]:
translation_result = model.generate_text(english_to_french_query)

Print the translation result

In [15]:
print(translation_result)

Output: Si nous n'augmentons pas considérablement l'utilisation de la technologie intelligente, nous n'atteindrons pas nos objectifs.



<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!

You learned how to translate multiple languages with `ibm/granite-4-h-small` model on watsonx. 

Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors and Maintainers

**Mateusz Szewczyk (Former)**, Software Engineer at IBM watsonx.ai

**Rafał Chrzanowski**, Software Engineer at IBM watsonx.ai

**Karol Zmorski**, Software Engineer at IBM watsonx.ai

Copyright © 2024-2026 IBM. This notebook and its source code are released under the terms of the MIT License.